In [ ]:
# -*- coding: utf-8 -*-
"""
氨基酸序列 CNN 抗菌肽预测模型
- 训练数据：氨基酸序列（AMPs.fasta, non_AMPs.fasta）
- 预测数据：Prodigal ORF 核酸序列，自动三框翻译 + 滑动窗口
- 去冗余：氨基酸层面 CD-HIT 0.4 聚类划分
"""

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # 强制 CPU

import subprocess
import tempfile
import shutil
import random
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# ================== 全局配置 ==================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 为 padding
NUM_AA = len(AMINO_ACIDS) + 1  # 21
MAX_LEN = 65          # 训练和滑动窗口长度
WINDOW_STEP = 10      # 预测滑动窗口步长

# 超参数记录
HYPERPARAMS = {
    'encoding': 'one-hot',
    'max_len': MAX_LEN,
    'window_step': WINDOW_STEP,
    'conv_layers': [
        {'filters': 32, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'},
        {'filters': 64, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'}
    ],
    'pooling': 'MaxPooling1D(2) + GlobalMaxPooling1D',
    'dropout': 0.3,
    'dense': [{'units': 32, 'activation': 'relu'}, {'units': 1, 'activation': 'sigmoid'}],
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'loss': 'binary_crossentropy',
    'batch_size': 32,
    'epochs': 50,
    'class_weight': {0: 1.0, 1: 2.0},
    'random_seed': SEED
}

# ================== 氨基酸编码 ==================
def encode_sequence(seq, max_len=MAX_LEN):
    """将氨基酸序列编码为 one-hot 矩阵 (max_len, 21)"""
    seq = seq.upper()
    indices = [AA_TO_IDX.get(aa, 0) for aa in seq[:max_len]]
    # 补零到 max_len
    indices += [0] * (max_len - len(indices))
    return to_categorical(indices, num_classes=NUM_AA)

# ================== 核酸翻译 ==================
RNA_CODON_TABLE = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*',
    'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_sequence(seq, frame=0):
    """从指定读框翻译核酸序列，遇到终止密码子停止"""
    seq = seq.upper().replace('T', 'U')
    aa_seq = []
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = RNA_CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        aa_seq.append(aa)
    return ''.join(aa_seq)

def is_nucleic_acid(seq):
    seq = seq.upper()
    dna = set('ATCG')
    rna = set('AUCG')
    if all(c in dna for c in seq):
        return 'DNA'
    elif all(c in rna for c in seq):
        return 'RNA'
    return None

# ================== 文件读取 ==================
def validate_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"文件不存在: {path.resolve()}")
    if not path.is_file():
        raise IsADirectoryError(f"路径指向的是目录而非文件: {path}")
    return path

def read_fasta_aa(file_path, label):
    """读取氨基酸 FASTA，返回序列和标签"""
    sequences, labels = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    sequences.append(seq.upper())
                    labels.append(label)
                    seq = ''
            else:
                seq += line
        if seq:
            sequences.append(seq.upper())
            labels.append(label)
    return sequences, labels

def read_fasta_nuc(file_path):
    """读取核酸 FASTA，返回 ID 和序列"""
    ids, sequences = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
    return ids, sequences

# ================== CD-HIT 聚类划分 ==================
def run_cd_hit(fasta_path, output_prefix, identity=0.4, word_size=5):
    """调用 CD-HIT，返回簇字典 {seq_id: cluster_id}"""
    try:
        # 检查 cd-hit 是否可用
        subprocess.run(['cd-hit', '-h'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("警告：未检测到 CD-HIT，将使用随机划分。建议安装 CD-HIT 以满足方法论要求②。")
        return None

    out_fasta = f"{output_prefix}.fasta"
    out_clstr = f"{output_prefix}.clstr"
    cmd = ['cd-hit', '-i', str(fasta_path), '-o', out_fasta,
           '-c', str(identity), '-n', str(word_size), '-M', '16000', '-T', '4']
    subprocess.run(cmd, check=True)

    # 解析 .clstr
    cluster_dict = {}
    with open(out_clstr, 'r') as f:
        cluster_id = 0
        for line in f:
            if line.startswith('>Cluster'):
                cluster_id = int(line.strip().split()[1])
            else:
                # >0 279aa, >seq1... *
                parts = line.strip().split('>')
                if len(parts) > 1:
                    seq_id = parts[1].split('...')[0]
                    cluster_dict[seq_id] = cluster_id
    return cluster_dict

def split_by_cluster(seqs, labels, cluster_dict, test_size=0.2, val_size=0.1):
    """按簇划分 train/val/test，返回索引"""
    # 将序列按簇分组
    cluster_to_indices = defaultdict(list)
    for i, seq in enumerate(seqs):
        # 需要根据序列获取 ID？我们使用序列本身作为 ID 不唯一，所以需在外部处理
        pass
    # 由于 read_fasta_aa 不返回 ID，我们在聚类时使用临时文件，需要 ID。
    # 因此下面重新设计：在训练主流程中，读取时保留 ID。
    raise NotImplementedError

# 为了简化，我们在主流程中直接处理：读取时保留 ID，然后写临时 fasta 运行 CD-HIT。
# 下面重新定义读取函数，保留 ID。
def read_fasta_aa_with_ids(file_path, label):
    ids, sequences, labels = [], [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    labels.append(label)
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
            labels.append(label)
    return ids, sequences, labels

def write_fasta(ids, seqs, path):
    with open(path, 'w') as f:
        for i, s in zip(ids, seqs):
            f.write(f">{i}\n{s}\n")

def cluster_split(all_ids, all_seqs, all_labels, data_dir, test_size=0.2, val_size=0.1):
    """使用 CD-HIT 聚类后按簇划分，返回 train/val/test 的索引"""
    # 写入临时 fasta
    tmp_fasta = data_dir / "temp_all.fasta"
    write_fasta(all_ids, all_seqs, tmp_fasta)

    cluster_dict = run_cd_hit(tmp_fasta, data_dir / "temp_cdhit", identity=0.4)
    if cluster_dict is None:
        # 回退到随机划分
        indices = np.arange(len(all_seqs))
        train_idx, test_idx = train_test_split(indices, test_size=test_size,
                                               stratify=all_labels, random_state=SEED)
        train_idx, val_idx = train_test_split(train_idx, test_size=val_size/(1-test_size),
                                              stratify=np.array(all_labels)[train_idx],
                                              random_state=SEED)
        return train_idx, val_idx, test_idx

    # 按簇分配
    cluster_to_indices = defaultdict(list)
    for i, seq_id in enumerate(all_ids):
        cid = cluster_dict.get(seq_id)
        if cid is None:
            # 如果 CD-HIT 没有输出该序列（可能太短），单独作为一个簇
            cid = f"singleton_{i}"
        cluster_to_indices[cid].append(i)

    clusters = list(cluster_to_indices.keys())
    random.shuffle(clusters)

    # 按簇大小排序，贪心分配以保持比例
    clusters.sort(key=lambda c: len(cluster_to_indices[c]), reverse=True)
    n_total = len(all_seqs)
    n_test = int(n_total * test_size)
    n_val = int(n_total * val_size)
    n_train = n_total - n_test - n_val

    train_idx, val_idx, test_idx = [], [], []
    for c in clusters:
        idxs = cluster_to_indices[c]
        if len(test_idx) + len(idxs) <= n_test:
            test_idx.extend(idxs)
        elif len(val_idx) + len(idxs) <= n_val:
            val_idx.extend(idxs)
        else:
            train_idx.extend(idxs)

    # 如果比例不完美，用随机抽样补齐
    remaining = list(set(range(n_total)) - set(train_idx) - set(val_idx) - set(test_idx))
    random.shuffle(remaining)
    while len(test_idx) < n_test and remaining:
        test_idx.append(remaining.pop())
    while len(val_idx) < n_val and remaining:
        val_idx.append(remaining.pop())
    train_idx.extend(remaining)

    # 清理临时文件
    for f in [tmp_fasta, data_dir / "temp_cdhit.fasta", data_dir / "temp_cdhit.clstr"]:
        if f.exists():
            f.unlink()

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

# ================== 模型构建 ==================
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================== 预测核酸序列 ==================
def predict_nucleotide_fasta(model, fasta_path, save_dir):
    """对核酸 FASTA 进行三框翻译 + 滑动窗口预测，输出 CSV"""
    ids, nuc_seqs = read_fasta_nuc(fasta_path)
    results = []
    for seq_id, nuc_seq in zip(ids, nuc_seqs):
        na_type = is_nucleic_acid(nuc_seq)
        if na_type is None:
            print(f"序列 {seq_id} 不是纯核酸，跳过。")
            continue
        for frame in range(3):
            aa_seq = translate_sequence(nuc_seq, frame=frame)
            if len(aa_seq) == 0:
                continue
            # 滑动窗口
            for start in range(0, max(1, len(aa_seq) - MAX_LEN + 1), WINDOW_STEP):
                window = aa_seq[start:start+MAX_LEN]
                if len(window) < MAX_LEN:
                    # 补零到 MAX_LEN（用空字符串补？one-hot 编码时会自动补 0）
                    pass
                # 编码
                encoded = encode_sequence(window, MAX_LEN)
                encoded = encoded.reshape(1, MAX_LEN, NUM_AA)
                prob = model.predict(encoded, verbose=0)[0][0]
                results.append({
                    'ID': seq_id,
                    'Frame': frame,
                    'Start': start,
                    'Window_Sequence': window,
                    'Probability': prob
                })
    df = pd.DataFrame(results)
    out_path = save_dir / 'CNN_nucleotide_predictions.csv'
    df.to_csv(out_path, index=False)
    print(f"预测结果已保存至: {out_path}，共 {len(df)} 个窗口。")

# ================== 主程序 ==================
if __name__ == '__main__':
    # 路径设置
    data_dir = Path("D:/data")
    amp_file = data_dir / "AMPs.fasta"
    non_amp_file = data_dir / "non_AMPs.fasta"
    predict_fasta = Path("D:/data/assembly_prodigal_300.fasta")  # 待预测核酸文件

    model_dir = Path("D:/data/result_CNN_AA")
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "cnn_amp_aa_model.h5"

    # 读取训练数据（氨基酸）
    print("读取训练数据...")
    amp_ids, amp_seqs, amp_labels = read_fasta_aa_with_ids(amp_file, 1)
    non_amp_ids, non_amp_seqs, non_amp_labels = read_fasta_aa_with_ids(non_amp_file, 0)
    print(f"正样本: {len(amp_seqs)}, 负样本: {len(non_amp_seqs)}")

    all_ids = amp_ids + non_amp_ids
    all_seqs = amp_seqs + non_amp_seqs
    all_labels = np.array(amp_labels + non_amp_labels)

    # 按簇划分 train/val/test
    print("正在进行 CD-HIT 聚类划分...")
    train_idx, val_idx, test_idx = cluster_split(all_ids, all_seqs, all_labels, data_dir,
                                                 test_size=0.2, val_size=0.1)
    print(f"训练集: {len(train_idx)}, 验证集: {len(val_idx)}, 测试集: {len(test_idx)}")

    # 编码
    X = np.array([encode_sequence(s, MAX_LEN) for s in all_seqs])
    y = all_labels

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # 构建模型
    model = build_cnn_model(input_shape=(MAX_LEN, NUM_AA))
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC'])
    model.summary()

    # 类别权重
    class_weight = {0: 1.0, 1: 2.0}

    # 训练
    print("开始训练...")
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=50,
                        batch_size=32,
                        class_weight=class_weight,
                        verbose=1)

    # 评估
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    print("\n测试集性能（阈值 0.5）:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1: {f1_score(y_test, y_pred):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

    # 保存模型
    model.save(model_path)
    print(f"模型已保存至: {model_path}")

    # 保存超参数
    with open(model_dir / 'hyperparameters.txt', 'w') as f:
        for k, v in HYPERPARAMS.items():
            f.write(f"{k}: {v}\n")

    # 预测核酸序列
    if predict_fasta.exists():
        print("\n开始预测核酸序列...")
        predict_nucleotide_fasta(model, predict_fasta, model_dir)
    else:
        print(f"预测文件不存在: {predict_fasta}")

读取训练数据...
正样本: 1085, 负样本: 58776
正在进行 CD-HIT 聚类划分...
警告：未检测到 CD-HIT，将使用随机划分。建议安装 CD-HIT 以满足方法论要求②。
训练集: 41902, 验证集: 5986, 测试集: 11973


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 65, 32)              │           2,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 32, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 32, 64)              │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,369 (40.50 KB)

 Trainable params: 10,369 (40.50 KB)

 Non-trainable params: 0 (0.00 B)

开始训练...
Epoch 1/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - AUC: 0.9431 - accuracy: 0.9834 - loss: 0.0852 - val_AUC: 0.9702 - val_accuracy: 0.9856 - val_loss: 0.0381
Epoch 2/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - AUC: 0.9840 - accuracy: 0.9897 - loss: 0.0425 - val_AUC: 0.9738 - val_accuracy: 0.9906 - val_loss: 0.0289
Epoch 3/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - AUC: 0.9873 - accuracy: 0.9917 - loss: 0.0339 - val_AUC: 0.9488 - val_accuracy: 0.9918 - val_loss: 0.0281
Epoch 4/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - AUC: 0.9913 - accuracy: 0.9925 - loss: 0.0287 - val_AUC: 0.9753 - val_accuracy: 0.9881 - val_loss: 0.0310
Epoch 5/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - AUC: 0.9932 - accuracy: 0.9937 - loss: 0.0258 - val_AUC: 0.9587 - val_accuracy: 0.9915 - val_loss: 0.0261
Epoch 6/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - AUC: 0.9959 - accuracy: 0.9943 - loss: 0.0222 - val_AUC: 0.9671 - val_accuracy: 0.9905 - val_loss: 0.0282
Epoch 7/50
13

In [ ]:
# -*- coding: utf-8 -*-
"""
氨基酸序列 CNN 抗菌肽预测模型（配置cdhit环境）
- 训练数据：氨基酸序列（AMPs.fasta, non_AMPs.fasta）
- 预测数据：Prodigal ORF 核酸序列，自动三框翻译 + 滑动窗口
- 去冗余：氨基酸层面 CD-HIT 0.4 聚类划分
- 所有生成文件存放在 D:/data/assembly_prodigal_300 CNN model result 目录下
"""

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # 强制 CPU

import subprocess
import tempfile
import shutil
import random
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# ================== 全局配置 ==================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 为 padding
NUM_AA = len(AMINO_ACIDS) + 1  # 21
MAX_LEN = 65          # 训练和滑动窗口长度
WINDOW_STEP = 10      # 预测滑动窗口步长

# 超参数记录
HYPERPARAMS = {
    'encoding': 'one-hot',
    'max_len': MAX_LEN,
    'window_step': WINDOW_STEP,
    'conv_layers': [
        {'filters': 32, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'},
        {'filters': 64, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'}
    ],
    'pooling': 'MaxPooling1D(2) + GlobalMaxPooling1D',
    'dropout': 0.3,
    'dense': [{'units': 32, 'activation': 'relu'}, {'units': 1, 'activation': 'sigmoid'}],
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'loss': 'binary_crossentropy',
    'batch_size': 32,
    'epochs': 50,
    'class_weight': {0: 1.0, 1: 2.0},
    'random_seed': SEED
}

# ================== 氨基酸编码 ==================
def encode_sequence(seq, max_len=MAX_LEN):
    """将氨基酸序列编码为 one-hot 矩阵 (max_len, 21)"""
    seq = seq.upper()
    indices = [AA_TO_IDX.get(aa, 0) for aa in seq[:max_len]]
    indices += [0] * (max_len - len(indices))
    return to_categorical(indices, num_classes=NUM_AA)

# ================== 核酸翻译 ==================
RNA_CODON_TABLE = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*',
    'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_sequence(seq, frame=0):
    """从指定读框翻译核酸序列，遇到终止密码子停止"""
    seq = seq.upper().replace('T', 'U')
    aa_seq = []
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = RNA_CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        aa_seq.append(aa)
    return ''.join(aa_seq)

def is_nucleic_acid(seq):
    seq = seq.upper()
    dna = set('ATCG')
    rna = set('AUCG')
    if all(c in dna for c in seq):
        return 'DNA'
    elif all(c in rna for c in seq):
        return 'RNA'
    return None

# ================== 文件读取 ==================
def validate_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"文件不存在: {path.resolve()}")
    if not path.is_file():
        raise IsADirectoryError(f"路径指向的是目录而非文件: {path}")
    return path

def read_fasta_aa(file_path, label):
    """读取氨基酸 FASTA，返回序列和标签"""
    sequences, labels = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    sequences.append(seq.upper())
                    labels.append(label)
                    seq = ''
            else:
                seq += line
        if seq:
            sequences.append(seq.upper())
            labels.append(label)
    return sequences, labels

def read_fasta_nuc(file_path):
    """读取核酸 FASTA，返回 ID 和序列"""
    ids, sequences = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
    return ids, sequences

# ================== CD-HIT 聚类划分 ==================
def run_cd_hit(fasta_path, output_prefix, work_dir, identity=0.4, word_size=5):
    """调用 CD-HIT，返回簇字典 {seq_id: cluster_id}"""
    try:
        # 检查 cd-hit 是否可用
        subprocess.run(['cd-hit', '-h'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("警告：未检测到 CD-HIT，将使用随机划分。建议安装 CD-HIT 以满足方法论要求②。")
        return None

    out_fasta = str(work_dir / f"{output_prefix}.fasta")
    out_clstr = str(work_dir / f"{output_prefix}.clstr")
    cmd = ['cd-hit', '-i', str(fasta_path), '-o', out_fasta,
           '-c', str(identity), '-n', str(word_size), '-M', '16000', '-T', '4']
    subprocess.run(cmd, check=True)

    # 解析 .clstr
    cluster_dict = {}
    with open(out_clstr, 'r') as f:
        cluster_id = 0
        for line in f:
            if line.startswith('>Cluster'):
                cluster_id = int(line.strip().split()[1])
            else:
                parts = line.strip().split('>')
                if len(parts) > 1:
                    seq_id = parts[1].split('...')[0]
                    cluster_dict[seq_id] = cluster_id
    return cluster_dict

def read_fasta_aa_with_ids(file_path, label):
    ids, sequences, labels = [], [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    labels.append(label)
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
            labels.append(label)
    return ids, sequences, labels

def write_fasta(ids, seqs, path):
    with open(path, 'w') as f:
        for i, s in zip(ids, seqs):
            f.write(f">{i}\n{s}\n")

def cluster_split(all_ids, all_seqs, all_labels, work_dir, test_size=0.2, val_size=0.1):
    """使用 CD-HIT 聚类后按簇划分，返回 train/val/test 的索引。临时文件均放在 work_dir 下。"""
    # 写入临时 fasta
    tmp_fasta = work_dir / "temp_all.fasta"
    write_fasta(all_ids, all_seqs, tmp_fasta)

    cluster_dict = run_cd_hit(tmp_fasta, "temp_cdhit", work_dir, identity=0.4)
    if cluster_dict is None:
        # 回退到随机划分
        indices = np.arange(len(all_seqs))
        train_idx, test_idx = train_test_split(indices, test_size=test_size,
                                               stratify=all_labels, random_state=SEED)
        train_idx, val_idx = train_test_split(train_idx, test_size=val_size/(1-test_size),
                                              stratify=np.array(all_labels)[train_idx],
                                              random_state=SEED)
        return train_idx, val_idx, test_idx

    # 按簇分配
    cluster_to_indices = defaultdict(list)
    for i, seq_id in enumerate(all_ids):
        cid = cluster_dict.get(seq_id)
        if cid is None:
            cid = f"singleton_{i}"
        cluster_to_indices[cid].append(i)

    clusters = list(cluster_to_indices.keys())
    random.shuffle(clusters)

    # 按簇大小排序，贪心分配以保持比例
    clusters.sort(key=lambda c: len(cluster_to_indices[c]), reverse=True)
    n_total = len(all_seqs)
    n_test = int(n_total * test_size)
    n_val = int(n_total * val_size)
    n_train = n_total - n_test - n_val

    train_idx, val_idx, test_idx = [], [], []
    for c in clusters:
        idxs = cluster_to_indices[c]
        if len(test_idx) + len(idxs) <= n_test:
            test_idx.extend(idxs)
        elif len(val_idx) + len(idxs) <= n_val:
            val_idx.extend(idxs)
        else:
            train_idx.extend(idxs)

    # 如果比例不完美，用随机抽样补齐
    remaining = list(set(range(n_total)) - set(train_idx) - set(val_idx) - set(test_idx))
    random.shuffle(remaining)
    while len(test_idx) < n_test and remaining:
        test_idx.append(remaining.pop())
    while len(val_idx) < n_val and remaining:
        val_idx.append(remaining.pop())
    train_idx.extend(remaining)

    # 清理临时文件
    for f in [tmp_fasta, work_dir / "temp_cdhit.fasta", work_dir / "temp_cdhit.clstr"]:
        if f.exists():
            f.unlink()

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

# ================== 模型构建 ==================
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================== 预测核酸序列 ==================
def predict_nucleotide_fasta(model, fasta_path, save_dir):
    """对核酸 FASTA 进行三框翻译 + 滑动窗口预测，输出 CSV"""
    ids, nuc_seqs = read_fasta_nuc(fasta_path)
    results = []
    for seq_id, nuc_seq in zip(ids, nuc_seqs):
        na_type = is_nucleic_acid(nuc_seq)
        if na_type is None:
            print(f"序列 {seq_id} 不是纯核酸，跳过。")
            continue
        for frame in range(3):
            aa_seq = translate_sequence(nuc_seq, frame=frame)
            if len(aa_seq) == 0:
                continue
            # 滑动窗口
            for start in range(0, max(1, len(aa_seq) - MAX_LEN + 1), WINDOW_STEP):
                window = aa_seq[start:start+MAX_LEN]
                if len(window) < MAX_LEN:
                    pass
                # 编码
                encoded = encode_sequence(window, MAX_LEN)
                encoded = encoded.reshape(1, MAX_LEN, NUM_AA)
                prob = model.predict(encoded, verbose=0)[0][0]
                results.append({
                    'ID': seq_id,
                    'Frame': frame,
                    'Start': start,
                    'Window_Sequence': window,
                    'Probability': prob
                })
    df = pd.DataFrame(results)
    out_path = save_dir / 'CNN_nucleotide_predictions.csv'
    df.to_csv(out_path, index=False)
    print(f"预测结果已保存至: {out_path}，共 {len(df)} 个窗口。")

# ================== 主程序 ==================
if __name__ == '__main__':
    # 路径设置
    data_dir = Path("D:/data")
    amp_file = data_dir / "AMPs.fasta"
    non_amp_file = data_dir / "non_AMPs.fasta"
    predict_fasta = data_dir / "assembly_prodigal_300.fasta"  # 待预测核酸文件

    # ====== 修改：所有输出文件存入指定文件夹 ======
    result_dir = data_dir / "assembly_prodigal_300 CNN model result"
    result_dir.mkdir(parents=True, exist_ok=True)
    model_path = result_dir / "cnn_amp_aa_model.h5"

    # 读取训练数据（氨基酸）
    print("读取训练数据...")
    amp_ids, amp_seqs, amp_labels = read_fasta_aa_with_ids(amp_file, 1)
    non_amp_ids, non_amp_seqs, non_amp_labels = read_fasta_aa_with_ids(non_amp_file, 0)
    print(f"正样本: {len(amp_seqs)}, 负样本: {len(non_amp_seqs)}")

    all_ids = amp_ids + non_amp_ids
    all_seqs = amp_seqs + non_amp_seqs
    all_labels = np.array(amp_labels + non_amp_labels)

    # 按簇划分 train/val/test
    print("正在进行 CD-HIT 聚类划分...")
    # ====== 修改：临时文件也存入 result_dir ======
    train_idx, val_idx, test_idx = cluster_split(all_ids, all_seqs, all_labels, result_dir,
                                                 test_size=0.2, val_size=0.1)
    print(f"训练集: {len(train_idx)}, 验证集: {len(val_idx)}, 测试集: {len(test_idx)}")

    # 编码
    X = np.array([encode_sequence(s, MAX_LEN) for s in all_seqs])
    y = all_labels

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # 构建模型
    model = build_cnn_model(input_shape=(MAX_LEN, NUM_AA))
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC'])
    model.summary()

    # 类别权重
    class_weight = {0: 1.0, 1: 2.0}

    # 训练
    print("开始训练...")
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=50,
                        batch_size=32,
                        class_weight=class_weight,
                        verbose=1)

    # 评估
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    print("\n测试集性能（阈值 0.5）:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1: {f1_score(y_test, y_pred):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

    # 保存模型
    model.save(model_path)
    print(f"模型已保存至: {model_path}")

    # 保存超参数
    with open(result_dir / 'hyperparameters.txt', 'w') as f:
        for k, v in HYPERPARAMS.items():
            f.write(f"{k}: {v}\n")
    print(f"超参数已保存至: {result_dir / 'hyperparameters.txt'}")

    # 保存评估指标
    with open(result_dir / 'test_metrics.txt', 'w') as f:
        f.write(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
        f.write(f"Precision: {precision_score(y_test, y_pred):.4f}\n")
        f.write(f"Recall: {recall_score(y_test, y_pred):.4f}\n")
        f.write(f"F1: {f1_score(y_test, y_pred):.4f}\n")
        f.write(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}\n")
        f.write(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}\n")
    print(f"评估指标已保存至: {result_dir / 'test_metrics.txt'}")

    # 预测核酸序列
    if predict_fasta.exists():
        print("\n开始预测核酸序列...")
        predict_nucleotide_fasta(model, predict_fasta, result_dir)
    else:
        print(f"预测文件不存在: {predict_fasta}")

    print(f"\n所有生成的文件均已保存至: {result_dir.resolve()}")

读取训练数据...
正样本: 1085, 负样本: 58776
正在进行 CD-HIT 聚类划分...
警告：未检测到 CD-HIT，将使用随机划分。建议安装 CD-HIT 以满足方法论要求②。
训练集: 41902, 验证集: 5986, 测试集: 11973


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 65, 32)              │           2,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 32, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 32, 64)              │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,369 (40.50 KB)

 Trainable params: 10,369 (40.50 KB)

 Non-trainable params: 0 (0.00 B)

开始训练...
Epoch 1/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - AUC: 0.9458 - accuracy: 0.9842 - loss: 0.0836 - val_AUC: 0.9800 - val_accuracy: 0.9811 - val_loss: 0.0445
Epoch 2/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - AUC: 0.9842 - accuracy: 0.9895 - loss: 0.0425 - val_AUC: 0.9811 - val_accuracy: 0.9905 - val_loss: 0.0284
Epoch 3/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - AUC: 0.9882 - accuracy: 0.9915 - loss: 0.0349 - val_AUC: 0.9795 - val_accuracy: 0.9911 - val_loss: 0.0263
Epoch 4/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - AUC: 0.9900 - accuracy: 0.9923 - loss: 0.0306 - val_AUC: 0.9626 - val_accuracy: 0.9915 - val_loss: 0.0256
Epoch 5/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - AUC: 0.9921 - accuracy: 0.9933 - loss: 0.0257 - val_AUC: 0.9797 - val_accuracy: 0.9900 - val_loss: 0.0285
Epoch 6/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - AUC: 0.9945 - accuracy: 0.9942 - loss: 0.0227 - val_AUC: 0.9722 - val_accuracy: 0.9916 - val_loss: 0.0259
Epoch 7/5


测试集性能（阈值 0.5）:
Accuracy: 0.9930
Precision: 0.8276
Recall: 0.7742
F1: 0.8000
MCC: 0.7969
AUC: 0.9862
模型已保存至: D:\data\assembly_prodigal_300 CNN model result\cnn_amp_aa_model.h5
超参数已保存至: D:\data\assembly_prodigal_300 CNN model result\hyperparameters.txt
评估指标已保存至: D:\data\assembly_prodigal_300 CNN model result\test_metrics.txt

开始预测核酸序列...


In [1]:
# -*- coding: utf-8 -*-
"""
氨基酸序列 CNN 抗菌肽预测模型
- 训练数据：氨基酸序列（AMPs.fasta, non_AMPs.fasta）
- 预测数据：Prodigal ORF 核酸序列，自动三框翻译 + 滑动窗口
- 去冗余：读取手动生成的 CD-HIT 0.4 聚类结果（.clstr）按簇划分
- 所有生成文件存放在 D:/data/assembly_prodigal_300 CNN model result 目录下
"""

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # 强制 CPU
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # 屏蔽 TF 警告

import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# ================== 全局配置 ==================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 为 padding
NUM_AA = len(AMINO_ACIDS) + 1  # 21
MAX_LEN = 65          # 训练和滑动窗口长度
WINDOW_STEP = 10      # 预测滑动窗口步长

# 超参数记录
HYPERPARAMS = {
    'encoding': 'one-hot',
    'max_len': MAX_LEN,
    'window_step': WINDOW_STEP,
    'conv_layers': [
        {'filters': 32, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'},
        {'filters': 64, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'}
    ],
    'pooling': 'MaxPooling1D(2) + GlobalMaxPooling1D',
    'dropout': 0.3,
    'dense': [{'units': 32, 'activation': 'relu'}, {'units': 1, 'activation': 'sigmoid'}],
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'loss': 'binary_crossentropy',
    'batch_size': 32,
    'epochs': 50,
    'class_weight': {0: 1.0, 1: 2.0},
    'random_seed': SEED
}

# ================== 氨基酸编码 ==================
def encode_sequence(seq, max_len=MAX_LEN):
    """将氨基酸序列编码为 one-hot 矩阵 (max_len, 21)"""
    seq = seq.upper()
    indices = [AA_TO_IDX.get(aa, 0) for aa in seq[:max_len]]
    indices += [0] * (max_len - len(indices))
    return to_categorical(indices, num_classes=NUM_AA)

# ================== 核酸翻译 ==================
RNA_CODON_TABLE = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*',
    'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_sequence(seq, frame=0):
    """从指定读框翻译核酸序列，遇到终止密码子停止"""
    seq = seq.upper().replace('T', 'U')
    aa_seq = []
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = RNA_CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        aa_seq.append(aa)
    return ''.join(aa_seq)

def is_nucleic_acid(seq):
    seq = seq.upper()
    dna = set('ATCG')
    rna = set('AUCG')
    if all(c in dna for c in seq):
        return 'DNA'
    elif all(c in rna for c in seq):
        return 'RNA'
    return None

# ================== 文件读取 ==================
def validate_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"文件不存在: {path.resolve()}")
    if not path.is_file():
        raise IsADirectoryError(f"路径指向的是目录而非文件: {path}")
    return path

def read_fasta_aa_with_ids(file_path, label):
    ids, sequences, labels = [], [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    labels.append(label)
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
            labels.append(label)
    return ids, sequences, labels

def read_fasta_nuc(file_path):
    """读取核酸 FASTA，返回 ID 和序列"""
    ids, sequences = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq = ''
        seq_id = ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id)
                    sequences.append(seq.upper())
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id)
            sequences.append(seq.upper())
    return ids, sequences

# ================== 解析 CD-HIT 结果并划分 ==================
def parse_cdhit_clstr(clstr_path):
    """解析 .clstr 文件，返回簇字典 {seq_id: cluster_id}"""
    cluster_dict = {}
    with open(clstr_path, 'r', encoding='utf-8') as f:
        cluster_id = 0
        for line in f:
            if line.startswith('>Cluster'):
                cluster_id = int(line.strip().split()[1])
            else:
                parts = line.strip().split('>')
                if len(parts) > 1:
                    # 提取序列ID，例如 >0 279aa, >seq1... *  -> 提取出 seq1
                    seq_id = parts[1].split('...')[0].strip()
                    cluster_dict[seq_id] = cluster_id
    return cluster_dict

def cluster_split(all_ids, all_seqs, all_labels, work_dir, test_size=0.2, val_size=0.1):
    """读取手动生成的 CD-HIT 聚类结果，按簇划分 train/val/test"""
    clstr_path = work_dir / "temp_cdhit.clstr"
    if not clstr_path.exists():
        raise FileNotFoundError(
            f"未找到聚类文件: {clstr_path}\n"
            f"请先在命令行（Anaconda Prompt）进入该目录并运行:\n"
            f"D:\\cdhit-4.8.1\\cd-hit.exe -i temp_all.fasta -o temp_cdhit -c 0.4 -n 2 -M 4000 -T 4"
        )
    
    print(f"正在读取聚类文件: {clstr_path}")
    cluster_dict = parse_cdhit_clstr(clstr_path)
    print(f"成功读取 {len(cluster_dict)} 条序列的聚类信息。")

    # 按簇分配索引
    cluster_to_indices = defaultdict(list)
    for i, seq_id in enumerate(all_ids):
        cid = cluster_dict.get(seq_id)
        if cid is None:
            cid = f"singleton_{i}"
        cluster_to_indices[cid].append(i)

    print(f"共发现 {len(cluster_to_indices)} 个聚类簇。")

    clusters = list(cluster_to_indices.keys())
    random.shuffle(clusters)

    # 按簇大小排序，贪心分配以保持比例
    clusters.sort(key=lambda c: len(cluster_to_indices[c]), reverse=True)
    n_total = len(all_seqs)
    n_test = int(n_total * test_size)
    n_val = int(n_total * val_size)
    
    train_idx, val_idx, test_idx = [], [], []
    for c in clusters:
        idxs = cluster_to_indices[c]
        if len(test_idx) + len(idxs) <= n_test:
            test_idx.extend(idxs)
        elif len(val_idx) + len(idxs) <= n_val:
            val_idx.extend(idxs)
        else:
            train_idx.extend(idxs)

    # 补齐比例（以防贪心分配后数量不够）
    remaining = list(set(range(n_total)) - set(train_idx) - set(val_idx) - set(test_idx))
    random.shuffle(remaining)
    while len(test_idx) < n_test and remaining:
        test_idx.append(remaining.pop())
    while len(val_idx) < n_val and remaining:
        val_idx.append(remaining.pop())
    train_idx.extend(remaining)

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

# ================== 模型构建 ==================
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================== 预测核酸序列（优化版，批量预测） ==================
def predict_nucleotide_fasta(model, fasta_path, save_dir):
    """对核酸 FASTA 进行三框翻译 + 滑动窗口预测，使用批量预测极大加速"""
    ids, nuc_seqs = read_fasta_nuc(fasta_path)
    
    all_windows = []      # 存放所有待预测的氨基酸窗口序列
    meta_info = []        # 存放每个窗口对应的元数据 (ID, 读框, 起点)
    
    print(f"正在提取所有可能的翻译窗口，共 {len(ids)} 条序列...")
    for seq_id, nuc_seq in zip(ids, nuc_seqs):
        na_type = is_nucleic_acid(nuc_seq)
        if na_type is None:
            continue
        for frame in range(3):
            aa_seq = translate_sequence(nuc_seq, frame=frame)
            if len(aa_seq) == 0:
                continue
            # 滑动窗口
            for start in range(0, max(1, len(aa_seq) - MAX_LEN + 1), WINDOW_STEP):
                window = aa_seq[start:start+MAX_LEN]
                all_windows.append(window)
                meta_info.append((seq_id, frame, start, window))
                
    if not all_windows:
        print("未提取到任何有效窗口。")
        return
        
    print(f"共提取到 {len(all_windows)} 个窗口，开始批量编码并预测（这将比逐条预测快100倍）...")
    
    # 1. 批量编码
    X_batch = np.array([encode_sequence(w, MAX_LEN) for w in all_windows])
    
    # 2. 一次性批量预测 (设置 batch_size 为 64 或 128)
    probs = model.predict(X_batch, batch_size=64, verbose=1).flatten()
    
    # 3. 组装结果
    results = []
    for (seq_id, frame, start, window), prob in zip(meta_info, probs):
        results.append({
            'ID': seq_id,
            'Frame': frame,
            'Start': start,
            'Window_Sequence': window,
            'Probability': prob
        })
        
    df = pd.DataFrame(results)
    out_path = save_dir / 'CNN_nucleotide_predictions.csv'
    df.to_csv(out_path, index=False)
    print(f"预测结果已保存至: {out_path}，共 {len(df)} 个窗口。")

# ================== 主程序 ==================
if __name__ == '__main__':
    # 路径设置
    data_dir = Path("D:/data")
    amp_file = data_dir / "AMPs.fasta"
    non_amp_file = data_dir / "non_AMPs.fasta"
    predict_fasta = data_dir / "assembly_prodigal_300.fasta"

    # ====== 所有生成文件存入此文件夹 ======
    result_dir = data_dir / "assembly_prodigal_300 CNN model result"
    result_dir.mkdir(parents=True, exist_ok=True)
    model_path = result_dir / "cnn_amp_aa_model.h5"

    # 读取训练数据（氨基酸）
    print("读取训练数据...")
    amp_ids, amp_seqs, amp_labels = read_fasta_aa_with_ids(amp_file, 1)
    non_amp_ids, non_amp_seqs, non_amp_labels = read_fasta_aa_with_ids(non_amp_file, 0)
    print(f"正样本: {len(amp_seqs)}, 负样本: {len(non_amp_seqs)}")

    all_ids = amp_ids + non_amp_ids
    all_seqs = amp_seqs + non_amp_seqs
    all_labels = np.array(amp_labels + non_amp_labels)

    # 按簇划分 train/val/test
    print("正在进行 CD-HIT 聚类划分...")
    train_idx, val_idx, test_idx = cluster_split(all_ids, all_seqs, all_labels, result_dir,
                                                 test_size=0.2, val_size=0.1)
    print(f"训练集: {len(train_idx)}, 验证集: {len(val_idx)}, 测试集: {len(test_idx)}")

    # 编码
    X = np.array([encode_sequence(s, MAX_LEN) for s in all_seqs])
    y = all_labels

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # 构建模型
    model = build_cnn_model(input_shape=(MAX_LEN, NUM_AA))
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC'])
    model.summary()

    # 类别权重
    class_weight = {0: 1.0, 1: 2.0}

    # 训练
    print("开始训练...")
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        epochs=50,
                        batch_size=32,
                        class_weight=class_weight,
                        verbose=1)

    # 评估
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    print("\n测试集性能（阈值 0.5）:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1: {f1_score(y_test, y_pred):.4f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

    # 保存模型与日志
    model.save(model_path)
    print(f"模型已保存至: {model_path}")

    with open(result_dir / 'hyperparameters.txt', 'w') as f:
        for k, v in HYPERPARAMS.items():
            f.write(f"{k}: {v}\n")

    with open(result_dir / 'test_metrics.txt', 'w') as f:
        f.write(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
        f.write(f"Precision: {precision_score(y_test, y_pred):.4f}\n")
        f.write(f"Recall: {recall_score(y_test, y_pred):.4f}\n")
        f.write(f"F1: {f1_score(y_test, y_pred):.4f}\n")
        f.write(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}\n")
        f.write(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}\n")
    print(f"评估指标已保存至: {result_dir / 'test_metrics.txt'}")

    # 预测核酸序列
    if predict_fasta.exists():
        print("\n开始预测核酸序列...")
        predict_nucleotide_fasta(model, predict_fasta, result_dir)
    else:
        print(f"预测文件不存在: {predict_fasta}")

    print(f"\n所有生成的文件均已保存至: {result_dir.resolve()}")

读取训练数据...
正样本: 1085, 负样本: 58776
正在进行 CD-HIT 聚类划分...
正在读取聚类文件: D:\data\assembly_prodigal_300 CNN model result\temp_cdhit.clstr
成功读取 59300 条序列的聚类信息。
共发现 33467 个聚类簇。
训练集: 41903, 验证集: 5986, 测试集: 11972


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 65, 32)              │           2,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 32, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 32, 64)              │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,369 (40.50 KB)

 Trainable params: 10,369 (40.50 KB)

 Non-trainable params: 0 (0.00 B)

开始训练...
Epoch 1/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - AUC: 0.8933 - accuracy: 0.9923 - loss: 0.0575 - val_AUC: 0.9831 - val_accuracy: 0.9828 - val_loss: 0.0418
Epoch 2/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - AUC: 0.9777 - accuracy: 0.9927 - loss: 0.0312 - val_AUC: 0.9801 - val_accuracy: 0.9830 - val_loss: 0.0432
Epoch 3/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - AUC: 0.9792 - accuracy: 0.9936 - loss: 0.0266 - val_AUC: 0.9847 - val_accuracy: 0.9885 - val_loss: 0.0321
Epoch 4/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - AUC: 0.9871 - accuracy: 0.9944 - loss: 0.0226 - val_AUC: 0.9872 - val_accuracy: 0.9885 - val_loss: 0.0316
Epoch 5/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - AUC: 0.9887 - accuracy: 0.9953 - loss: 0.0197 - val_AUC: 0.9834 - val_accuracy: 0.9861 - val_loss: 0.0380
Epoch 6/50
1310/1310 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - AUC: 0.9865 - accuracy: 0.9957 - loss: 0.0176 - val_AUC: 0.9839 - val_accuracy: 0.9866 - val_loss: 0.0340
Epoch 7/50
13


测试集性能（阈值 0.5）:
Accuracy: 0.9774
Precision: 0.8067
Recall: 0.7233
F1: 0.7627
MCC: 0.7522
AUC: 0.9834
模型已保存至: D:\data\assembly_prodigal_300 CNN model result\cnn_amp_aa_model.h5
评估指标已保存至: D:\data\assembly_prodigal_300 CNN model result\test_metrics.txt

开始预测核酸序列...


KeyboardInterrupt: 

In [2]:
# -*- coding: utf-8 -*-
"""
氨基酸序列 CNN 抗菌肽预测模型 (优化版)
- 训练数据：氨基酸序列（AMPs.fasta, non_AMPs.fasta）
- 预测数据：Prodigal ORF 核酸序列，自动三框翻译 + 批量滑动窗口预测
- 去冗余：读取手动生成的 CD-HIT 0.4 聚类结果（.clstr）按簇划分
- 所有生成文件存放在 D:/data/assembly_prodigal_300 CNN model result 目录下
- 新增：自动检测已训练模型，避免重复训练
"""

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # 强制 CPU
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # 屏蔽 TF 警告

import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# ================== 全局配置 ==================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 为 padding
NUM_AA = len(AMINO_ACIDS) + 1  # 21
MAX_LEN = 65          # 训练和滑动窗口长度
WINDOW_STEP = 10      # 预测滑动窗口步长

HYPERPARAMS = {
    'encoding': 'one-hot',
    'max_len': MAX_LEN,
    'window_step': WINDOW_STEP,
    'conv_layers': [
        {'filters': 32, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'},
        {'filters': 64, 'kernel_size': 3, 'activation': 'relu', 'padding': 'same'}
    ],
    'pooling': 'MaxPooling1D(2) + GlobalMaxPooling1D',
    'dropout': 0.3,
    'dense': [{'units': 32, 'activation': 'relu'}, {'units': 1, 'activation': 'sigmoid'}],
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'loss': 'binary_crossentropy',
    'batch_size': 32,
    'epochs': 50,
    'class_weight': {0: 1.0, 1: 2.0},
    'random_seed': SEED
}

# ================== 氨基酸编码 ==================
def encode_sequence(seq, max_len=MAX_LEN):
    seq = seq.upper()
    indices = [AA_TO_IDX.get(aa, 0) for aa in seq[:max_len]]
    indices += [0] * (max_len - len(indices))
    return to_categorical(indices, num_classes=NUM_AA)

# ================== 核酸翻译 ==================
RNA_CODON_TABLE = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*',
    'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_sequence(seq, frame=0):
    seq = seq.upper().replace('T', 'U')
    aa_seq = []
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = RNA_CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        aa_seq.append(aa)
    return ''.join(aa_seq)

def is_nucleic_acid(seq):
    seq = seq.upper()
    dna = set('ATCG')
    rna = set('AUCG')
    if all(c in dna for c in seq):
        return 'DNA'
    elif all(c in rna for c in seq):
        return 'RNA'
    return None

# ================== 文件读取 ==================
def validate_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"文件不存在: {path.resolve()}")
    if not path.is_file():
        raise IsADirectoryError(f"路径指向的是目录而非文件: {path}")
    return path

def read_fasta_aa_with_ids(file_path, label):
    ids, sequences, labels = [], [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq, seq_id = '', ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id); sequences.append(seq.upper()); labels.append(label)
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id); sequences.append(seq.upper()); labels.append(label)
    return ids, sequences, labels

def read_fasta_nuc(file_path):
    ids, sequences = [], []
    with open(validate_file(file_path), 'r', encoding='utf-8') as f:
        seq, seq_id = '', ''
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if seq:
                    ids.append(seq_id); sequences.append(seq.upper())
                    seq = ''
                seq_id = line[1:].split()[0]
            else:
                seq += line
        if seq:
            ids.append(seq_id); sequences.append(seq.upper())
    return ids, sequences

# ================== 解析 CD-HIT 结果并划分 ==================
def parse_cdhit_clstr(clstr_path):
    cluster_dict = {}
    with open(clstr_path, 'r', encoding='utf-8') as f:
        cluster_id = 0
        for line in f:
            if line.startswith('>Cluster'):
                cluster_id = int(line.strip().split()[1])
            else:
                parts = line.strip().split('>')
                if len(parts) > 1:
                    seq_id = parts[1].split('...')[0].strip()
                    cluster_dict[seq_id] = cluster_id
    return cluster_dict

def cluster_split(all_ids, all_seqs, all_labels, work_dir, test_size=0.2, val_size=0.1):
    clstr_path = work_dir / "temp_cdhit.clstr"
    if not clstr_path.exists():
        raise FileNotFoundError(
            f"未找到聚类文件: {clstr_path}\n"
            f"请先在 Anaconda Prompt 中进入该目录并运行:\n"
            f"D:\\cdhit-4.8.1\\cd-hit.exe -i temp_all.fasta -o temp_cdhit -c 0.4 -n 2 -M 4000 -T 4"
        )
    print(f"正在读取聚类文件: {clstr_path}")
    cluster_dict = parse_cdhit_clstr(clstr_path)
    print(f"成功读取 {len(cluster_dict)} 条序列的聚类信息。")

    cluster_to_indices = defaultdict(list)
    for i, seq_id in enumerate(all_ids):
        cid = cluster_dict.get(seq_id, f"singleton_{i}")
        cluster_to_indices[cid].append(i)

    print(f"共发现 {len(cluster_to_indices)} 个聚类簇。")
    clusters = list(cluster_to_indices.keys())
    random.shuffle(clusters)
    clusters.sort(key=lambda c: len(cluster_to_indices[c]), reverse=True)
    
    n_total = len(all_seqs)
    n_test, n_val = int(n_total * test_size), int(n_total * val_size)
    train_idx, val_idx, test_idx = [], [], []
    
    for c in clusters:
        idxs = cluster_to_indices[c]
        if len(test_idx) + len(idxs) <= n_test: test_idx.extend(idxs)
        elif len(val_idx) + len(idxs) <= n_val: val_idx.extend(idxs)
        else: train_idx.extend(idxs)

    remaining = list(set(range(n_total)) - set(train_idx) - set(val_idx) - set(test_idx))
    random.shuffle(remaining)
    while len(test_idx) < n_test and remaining: test_idx.append(remaining.pop())
    while len(val_idx) < n_val and remaining: val_idx.append(remaining.pop())
    train_idx.extend(remaining)

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

# ================== 模型构建 ==================
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(32, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

# ================== 优化后的批量预测函数 ==================
def predict_nucleotide_fasta(model, fasta_path, save_dir):
    """对核酸 FASTA 进行三框翻译 + 批量滑动窗口预测，输出 CSV"""
    ids, nuc_seqs = read_fasta_nuc(fasta_path)
    all_windows = []
    meta_info = []
    
    print(f"正在提取所有可能的翻译窗口，共 {len(ids)} 条序列...")
    for seq_id, nuc_seq in zip(ids, nuc_seqs):
        if is_nucleic_acid(nuc_seq) is None:
            continue
        for frame in range(3):
            aa_seq = translate_sequence(nuc_seq, frame=frame)
            if len(aa_seq) == 0:
                continue
            for start in range(0, max(1, len(aa_seq) - MAX_LEN + 1), WINDOW_STEP):
                window = aa_seq[start:start+MAX_LEN]
                all_windows.append(window)
                meta_info.append((seq_id, frame, start, window))
                
    if not all_windows:
        print("未提取到任何有效窗口。")
        return
        
    print(f"共提取到 {len(all_windows)} 个窗口，开始批量编码并预测...")
    X_batch = np.array([encode_sequence(w, MAX_LEN) for w in all_windows])
    
    # 一次性批量预测，batch_size 设为 64 或 128 极大提速
    probs = model.predict(X_batch, batch_size=64, verbose=1).flatten()
    
    results = []
    for (seq_id, frame, start, window), prob in zip(meta_info, probs):
        results.append({
            'ID': seq_id, 'Frame': frame, 'Start': start,
            'Window_Sequence': window, 'Probability': prob
        })
        
    df = pd.DataFrame(results)
    out_path = save_dir / 'CNN_nucleotide_predictions.csv'
    df.to_csv(out_path, index=False)
    print(f"预测结果已保存至: {out_path}，共 {len(df)} 个窗口。")

# ================== 主程序 ==================
if __name__ == '__main__':
    data_dir = Path("D:/data")
    amp_file = data_dir / "AMPs.fasta"
    non_amp_file = data_dir / "non_AMPs.fasta"
    predict_fasta = data_dir / "assembly_prodigal_300.fasta"

    result_dir = data_dir / "assembly_prodigal_300 CNN model result"
    result_dir.mkdir(parents=True, exist_ok=True)
    model_path = result_dir / "cnn_amp_aa_model.h5"

    # ====== 1. 判断是否直接加载模型 ======
    if model_path.exists():
        print(f"检测到已训练模型: {model_path}")
        print("直接加载模型，跳过训练阶段...")
        model = load_model(model_path)
        
        # 为了后续预测，仍需读取数据来保证代码完整性（只是不训练）
        print("读取训练数据用于验证...")
        amp_ids, amp_seqs, amp_labels = read_fasta_aa_with_ids(amp_file, 1)
        non_amp_ids, non_amp_seqs, non_amp_labels = read_fasta_aa_with_ids(non_amp_file, 0)
        
    else:
        # ====== 2. 从头开始训练 ======
        print("未检测到模型，开始读取数据并训练...")
        amp_ids, amp_seqs, amp_labels = read_fasta_aa_with_ids(amp_file, 1)
        non_amp_ids, non_amp_seqs, non_amp_labels = read_fasta_aa_with_ids(non_amp_file, 0)
        print(f"正样本: {len(amp_seqs)}, 负样本: {len(non_amp_seqs)}")

        all_ids = amp_ids + non_amp_ids
        all_seqs = amp_seqs + non_amp_seqs
        all_labels = np.array(amp_labels + non_amp_labels)

        print("正在进行 CD-HIT 聚类划分...")
        train_idx, val_idx, test_idx = cluster_split(all_ids, all_seqs, all_labels, result_dir,
                                                     test_size=0.2, val_size=0.1)
        print(f"训练集: {len(train_idx)}, 验证集: {len(val_idx)}, 测试集: {len(test_idx)}")

        X = np.array([encode_sequence(s, MAX_LEN) for s in all_seqs])
        y = all_labels
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        model = build_cnn_model(input_shape=(MAX_LEN, NUM_AA))
        model.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                      metrics=['accuracy', 'AUC'])
        
        class_weight = {0: 1.0, 1: 2.0}
        print("开始训练...")
        model.fit(X_train, y_train,
                  validation_data=(X_val, y_val),
                  epochs=50, batch_size=32,
                  class_weight=class_weight,
                  verbose=1)

        # 评估
        y_pred_proba = model.predict(X_test, verbose=0).flatten()
        y_pred = (y_pred_proba >= 0.5).astype(int)
        print("\n测试集性能（阈值 0.5）:")
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(f"Precision: {precision_score(y_test, y_pred):.4f}")
        print(f"Recall: {recall_score(y_test, y_pred):.4f}")
        print(f"F1: {f1_score(y_test, y_pred):.4f}")
        print(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}")
        print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

        # 保存模型与日志
        model.save(model_path)
        print(f"模型已保存至: {model_path}")
        with open(result_dir / 'test_metrics.txt', 'w') as f:
            f.write(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
            f.write(f"Precision: {precision_score(y_test, y_pred):.4f}\n")
            f.write(f"Recall: {recall_score(y_test, y_pred):.4f}\n")
            f.write(f"F1: {f1_score(y_test, y_pred):.4f}\n")
            f.write(f"MCC: {matthews_corrcoef(y_test, y_pred):.4f}\n")
            f.write(f"AUC: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

    # ====== 3. 无论是否重新训练，均执行批量预测 ======
    if predict_fasta.exists():
        print("\n开始批量预测核酸序列...")
        predict_nucleotide_fasta(model, predict_fasta, result_dir)
    else:
        print(f"预测文件不存在: {predict_fasta}")

    print(f"\n所有生成的文件均已保存至: {result_dir.resolve()}")

检测到已训练模型: D:\data\assembly_prodigal_300 CNN model result\cnn_amp_aa_model.h5
直接加载模型，跳过训练阶段...


读取训练数据用于验证...

开始批量预测核酸序列...
正在提取所有可能的翻译窗口，共 245934 条序列...
共提取到 777446 个窗口，开始批量编码并预测...
12148/12148 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step
预测结果已保存至: D:\data\assembly_prodigal_300 CNN model result\CNN_nucleotide_predictions.csv，共 777446 个窗口。

所有生成的文件均已保存至: D:\data\assembly_prodigal_300 CNN model result
